# Oncology Antibody IP/Clinical Intelligence App — Data Analytics Agent (local prototype)

Phase 0/1 of the app plan (see `/memories/session/plan.md`): build and validate, **entirely locally**,
the data-analytics-agent logic that will later power the web app. Nothing here is pushed to GitHub yet.

**Structure of this notebook:**
1. **Phase 0 — Data ETL / trimming.** Read the raw curated CSVs (`ip_final_version3.csv`,
   `ip_final_version4.csv`, `clinical_final_version1.csv`) and write small "app-ready" copies that keep
   only the columns actually needed by (a) the basic landscape/comparison queries and (b) the
   whitespace/FTO/triangulation engine — dropping all raw patent full-text (`claims_fulltext`,
   `claims_excerpt`, snippet columns, description text, raw JSON paths). This shrinks the working data from
   >50MB to a few MB and means nothing raw/proprietary ever needs to leave this machine.
2. **Phase 1 — Query tool functions.** Deterministic pandas/DuckDB functions for "top-N", "sort/filter",
   "group-count", and cross-dataset comparison queries, plus a thin wrapper around
   `multitarget_locked_whitespace_workflow.MTW.run()` for 2x2/triangulation queries.
3. **Phase 1 — Data Analytics Agent.** An OpenAI function-calling layer with a "10-15yr oncology/biologics
   data analytics expert" persona that can ONLY call the vetted tool functions above (never freeform
   code/SQL, never sees raw rows — only the aggregated result each tool returns).
4. **Validation.** Runs the two example queries from the brief plus a comparison query and a
   2x2/triangulation query end-to-end.


## Phase 0 — Data ETL / trimming

Column whitelists below were derived by reading `multitarget_locked_whitespace_workflow.py` and
`locked_whitespace_workflow_csv_v3_param.py` line-by-line to find every column they actually reference
(see `/memories/session/plan.md` for the audit trail), unioned with the descriptive columns needed to
answer basic landscape/comparison questions (top-N sponsors, sort/filter by modality/country, etc.).
Everything else — `claims_fulltext`, `claims_excerpt`, `claim_*_snippets`, `description_text_chars`,
`patsnap_raw_json_path`, and other raw/debug free-text — is dropped. This is what makes the resulting
`input/app_data/*.csv` files small enough (and clean enough) to eventually ship with the app without
exposing the actual curated patent/trial text.


In [28]:
import os, sys, json, math, time
from pathlib import Path

import numpy as np
import pandas as pd
import duckdb

INPUT_DIR = Path(os.path.abspath("."))          # this notebook lives in input/
APP_DATA_DIR = INPUT_DIR / "app_data"
APP_DATA_DIR.mkdir(exist_ok=True)

RAW_IP_LANDSCAPE_CSV = INPUT_DIR / "ip_final_version3.csv"   # basic landscape/comparison tier
RAW_IP_WHITESPACE_CSV = INPUT_DIR / "ip_final_version4.csv"  # 2x2 / triangulation tier
RAW_CLINICAL_CSV = INPUT_DIR / "clinical_final_version1.csv" # used by both tiers

APP_IP_LANDSCAPE_CSV = APP_DATA_DIR / "ip_landscape_app.csv"
APP_IP_WHITESPACE_CSV = APP_DATA_DIR / "ip_whitespace_app.csv"
APP_CLINICAL_CSV = APP_DATA_DIR / "clinical_app.csv"

# --- IP whitelist -----------------------------------------------------------
# Columns actually read by the whitespace/FTO engine (build_ip_index + W._claims_grounded /
# W._is_locking_claim / W._is_use_combo / W._ip_apd), UNION with descriptive columns needed
# for basic landscape/comparison queries (sponsor/top-N, sort by country, modality, indication).
IP_WHITELIST = [
    # identifiers / descriptive
    "ip_master_id", "publication_number", "application_number", "authority", "title",
    "current_assignee", "filing_year", "application_date", "publication_date",
    "indications", "target", "target_harmonized", "target_class",
    "modality_code", "modality_only", "all_detected_modalities", "payload_short",
    "is_antibody", "n_targets", "targets_hgnc",
    # whitespace/FTO engine inputs (do NOT remove — required by MTW.run())
    "category", "claim_focus",
    "patsnap_claims_fulltext_status", "target_modality_crowding_proxy_basis",
    "claim_mentions_target", "claim_mentions_epitope_domain_or_competition",
    "independent_claim_count", "total_claim_count",
]

# --- Clinical whitelist ------------------------------------------------------
# clinical_final_version1.csv has no giant free-text blob columns, so trimming here is about
# dropping pure QA/audit-trail columns, not about file size. Whitespace engine (build_clin_index)
# only needs target_harmonized/target, modality_code, start_date, outcome_enriched/outcome,
# phase/ctgov_phase -- everything else below is for basic landscape/comparison queries.
CLINICAL_DROP = [
    "asset_confidence", "asset_verified", "asset_citation",
    "trial_confidence", "trial_verified", "notes",
    "_row_source", "nct_in_golden", "ctgov_fetch_status",
    "target_vocab_version", "targets_hgnc_original", "m_source_dataset",
]

# Integer-ID-like columns that the whitespace/FTO engine parses with strict `.isdigit()` checks
# (multitarget_locked_whitespace_workflow.py's W._ip_apd, etc). These must be written to the on-disk
# CSV as plain digit strings ("2010"), never as a stringified float ("2010.0") -- the latter silently
# fails `.isdigit()` and breaks date-cutoff gating with NO exception raised. This matters even when the
# raw source file itself already has the "2010.0" defect (confirmed present in ip_final_version4.csv,
# NOT present in ip_final_version3.csv) -- converting to a pandas nullable Int64 before to_csv repairs
# that pre-existing upstream defect as a side effect.
INT_LIKE_COLS = ["filing_year", "application_date", "publication_date",
                  "independent_claim_count", "total_claim_count", "n_targets"]

def trim_csv(src: Path, dst: Path, *, whitelist=None, drop=None):
    assert (whitelist is None) != (drop is None), "pass exactly one of whitelist/drop"
    header = pd.read_csv(src, nrows=0).columns.tolist()
    if whitelist is not None:
        missing = [c for c in whitelist if c not in header]
        if missing:
            print(f"  [warn] {src.name}: whitelist columns not found in file (skipped): {missing}")
        keep = [c for c in whitelist if c in header]
    else:
        keep = [c for c in header if c not in drop]
    t0 = time.time()
    df = pd.read_csv(src, usecols=keep, low_memory=False)
    out = df.copy()
    for c in INT_LIKE_COLS:
        if c in out.columns:
            out[c] = pd.to_numeric(out[c], errors="coerce").astype("Int64")
    out.to_csv(dst, index=False)
    src_mb = src.stat().st_size / 1e6
    dst_mb = dst.stat().st_size / 1e6
    print(f"{src.name}: {len(header)} cols -> {len(keep)} cols kept | "
          f"{out.shape[0]:,} rows | {src_mb:.1f}MB -> {dst_mb:.1f}MB | {time.time()-t0:.1f}s")
    return out

ip_landscape_df = trim_csv(RAW_IP_LANDSCAPE_CSV, APP_IP_LANDSCAPE_CSV, whitelist=IP_WHITELIST)
ip_whitespace_df = trim_csv(RAW_IP_WHITESPACE_CSV, APP_IP_WHITESPACE_CSV, whitelist=IP_WHITELIST)
clinical_df = trim_csv(RAW_CLINICAL_CSV, APP_CLINICAL_CSV, drop=CLINICAL_DROP)


ip_final_version3.csv: 72 cols -> 28 cols kept | 8,901 rows | 74.3MB -> 3.6MB | 0.2s
ip_final_version4.csv: 72 cols -> 28 cols kept | 4,005 rows | 39.3MB -> 1.7MB | 0.1s
clinical_final_version1.csv: 63 cols -> 51 cols kept | 19,357 rows | 19.6MB -> 16.1MB | 0.3s


### Phase 0 regression check — confirm the trimmed CSVs give identical 2x2/whitespace results

Before trusting the trimmed files for anything, prove `MTW.run()` (the whitespace/FTO engine) produces
byte-identical numbers whether it reads the original raw CSVs or the trimmed `app_data/*.csv` copies.


In [29]:
import sys, tempfile
sys.path.insert(0, str(INPUT_DIR))
import multitarget_locked_whitespace_workflow as MTW

# NOTE: ip_final_version4.csv has a pre-existing upstream defect (application_date/filing_year stored as
# "2010.0" instead of "2010" -- see the diagnostic cells below), which our Phase 0 trim_csv() repairs via
# an Int64 cast. That means a byte-for-byte "raw vs trimmed" comparison is NO LONGER a valid regression
# check for the whitespace/2x2 tier: the raw file is silently broken (every node reports
# x_resolved=False/UNRESOLVED_no_grounded_patents) and the trimmed file is intentionally different
# (repaired). So instead we independently repair a copy of the raw file here (self-contained, not reusing
# trim_csv) and confirm OUR ETL's trimmed output matches that independent repair -- proving the repair
# logic is correct and reproducible, not just "different from a known-broken baseline".
_int_like = ["filing_year", "application_date", "publication_date",
             "independent_claim_count", "total_claim_count", "n_targets"]
with tempfile.TemporaryDirectory() as tmp_raw, tempfile.TemporaryDirectory() as tmp_trim, \
     tempfile.NamedTemporaryFile(suffix=".csv", delete=False) as tmp_repaired_f:
    _raw_df = pd.read_csv(RAW_IP_WHITESPACE_CSV, low_memory=False)
    for c in _int_like:
        if c in _raw_df.columns:
            _raw_df[c] = pd.to_numeric(_raw_df[c], errors="coerce").astype("Int64")
    _raw_df.to_csv(tmp_repaired_f.name, index=False)

    rows_raw_repaired = MTW.run(
        ip_csv=tmp_repaired_f.name, clin_csv=str(RAW_CLINICAL_CSV),
        as_of=MTW.DEF_AS_OF, indication_terms=[t.strip().lower() for t in MTW.DEF_INDICATION.split(",")],
        outdir=tmp_raw, key_col=MTW.DEF_KEY_COL, mod_col=MTW.DEF_MOD_COL,
        by_modality=False, sample_n=40, seed=7,
    )
    rows_trim = MTW.run(
        ip_csv=str(APP_IP_WHITESPACE_CSV), clin_csv=str(APP_CLINICAL_CSV),
        as_of=MTW.DEF_AS_OF, indication_terms=[t.strip().lower() for t in MTW.DEF_INDICATION.split(",")],
        outdir=tmp_trim, key_col=MTW.DEF_KEY_COL, mod_col=MTW.DEF_MOD_COL,
        by_modality=False, sample_n=40, seed=7,
    )
os.remove(tmp_repaired_f.name)

raw_by_key = {r["target_harmonized"]: r for r in rows_raw_repaired}
trim_by_key = {r["target_harmonized"]: r for r in rows_trim}
assert set(raw_by_key) == set(trim_by_key), "node set differs between independently-repaired-raw and trimmed run!"
mismatches = []
for k, r in raw_by_key.items():
    t = trim_by_key[k]
    for field in ("quadrant", "clinical_performance_Y", "epitope_crowding_X", "n_trials",
                  "patents_grounded", "x_resolved", "y_resolved"):
        if r[field] != t[field]:
            mismatches.append((k, field, r[field], t[field]))
n_resolved = sum(1 for r in raw_by_key.values() if r.get("x_resolved"))
print(f"Compared {len(raw_by_key)} nodes. {n_resolved}/{len(raw_by_key)} have x_resolved=True in the repaired baseline.")
if mismatches:
    print(f"MISMATCHES ({len(mismatches)}):")
    for m in mismatches[:20]:
        print("  ", m)
else:
    print("PASS — trimmed app_data CSVs produce identical whitespace/2x2 results (incl. x_resolved) to an "
          "independently date-repaired copy of the raw CSVs.")



MULTI-TARGET white-space (deterministic) — as of 2026-05-05 — mode: target (modality-collapsed)
    [ip] 4005 patent rows -> 1139 harmonized targets
    [clin] 19357 trial rows -> 1691 harmonized targets
    [sample] 40 nodes (seed=7)
    [out] /var/folders/k_/q9w49kk53zg7v019tx90gbjh0000gn/T/tmpz4o7n3a9/master_multitarget_whitespace.csv (40 nodes)
    [out] /var/folders/k_/q9w49kk53zg7v019tx90gbjh0000gn/T/tmpz4o7n3a9/MAP_2x2_REPORT.md
    [out] /var/folders/k_/q9w49kk53zg7v019tx90gbjh0000gn/T/tmpz4o7n3a9/multitarget_2x2.png
done.

MULTI-TARGET white-space (deterministic) — as of 2026-05-05 — mode: target (modality-collapsed)
    [ip] 4005 patent rows -> 1139 harmonized targets
    [clin] 19357 trial rows -> 1691 harmonized targets
    [sample] 40 nodes (seed=7)
    [out] /var/folders/k_/q9w49kk53zg7v019tx90gbjh0000gn/T/tmpjya8hjtd/master_multitarget_whitespace.csv (40 nodes)
    [out] /var/folders/k_/q9w49kk53zg7v019tx90gbjh0000gn/T/tmpjya8hjtd/MAP_2x2_REPORT.md


Compared 40 nodes. 19/40 have x_resolved=True in the repaired baseline.
PASS — trimmed app_data CSVs produce identical whitespace/2x2 results (incl. x_resolved) to an independently date-repaired copy of the raw CSVs.


    [out] /var/folders/k_/q9w49kk53zg7v019tx90gbjh0000gn/T/tmpjya8hjtd/multitarget_2x2.png
done.


## Phase 1 — Query tool functions

First, inspect the vocabulary (distinct values of the columns the agent will filter/group on) so the
function-calling schema can constrain the LLM to real values instead of letting it invent filters.


In [30]:
print("IP modality_code:", ip_landscape_df["modality_code"].value_counts(dropna=False).to_dict())
print()
print("IP authority (patent office / country proxy):", ip_landscape_df["authority"].value_counts(dropna=False).head(15).to_dict())
print()
print("IP current_assignee sample:", ip_landscape_df["current_assignee"].dropna().iloc[:3].tolist())
print()
print("Clinical modality_code:", clinical_df["modality_code"].value_counts(dropna=False).to_dict())
print()
print("Clinical in_scope dtype:", clinical_df["in_scope"].dtype, clinical_df["in_scope"].value_counts(dropna=False).to_dict())
print()
print("Clinical outcome:", clinical_df["outcome"].value_counts(dropna=False).to_dict())
print()
print("Clinical phase:", clinical_df["phase"].value_counts(dropna=False).head(10).to_dict())


IP modality_code: {'MAB': 6274, 'BISPECIFIC': 2005, 'ADC': 586, 'PROTAC': 26, 'BiTE': 10}

IP authority (patent office / country proxy): {'WO': 8885, nan: 16}

IP current_assignee sample: ['GLAXO GROUP LIMITED|DUFFIELD, STEPHEN|ENEVER, CAROLYN|LIU, HAIQUN|SCHON, OLIVER|SEPP, ARMIN|STOOP, ALLART ADRIAAN', 'GLAXO GROUP LIMITED|HAMBLIN, PAUL, ANDREW|PARMAR, RADHA, SHAH|WHITE, JOHN', 'IACOBELLI STEFANO']

Clinical modality_code: {'MAB': 12452, 'UNCLASSIFIED': 2514, 'ADC': 1554, 'CAR-T': 1472, 'BISPECIFIC': 1064, 'OUT_OF_SCOPE': 205, 'BiTE': 63, 'RADIOLIGAND': 16, 'PROTAC': 12, 'MULTISPECIFIC': 3, 'DAC': 1, 'ANTISENSE': 1}

Clinical in_scope dtype: object {True: 16648, False: 2684, nan: 25}

Clinical outcome: {'ongoing': 6771, 'pending': 5143, 'unclear': 3016, 'terminated': 2640, 'positive': 667, 'approved': 665, 'negative': 455}

Clinical phase: {'PHASE2': 7645, 'PHASE1': 4337, 'PHASE1|PHASE2': 2727, nan: 1768, 'PHASE3': 1760, 'EARLY_PHASE1': 401, 'PHASE2|PHASE3': 227, 'PHASE4': 161, '2': 

**Data-quality findings that shape the tool functions below (important — avoid hallucinated answers):**
- `authority` (the closest thing to "source country") is **8,885/8,901 = "WO"** (PCT international
  filing) — this dataset does not actually carry per-country patent office breakdown. A query like *"sort
  ADC patents by source country"* will technically run, but will return an almost-degenerate single-value
  result. The tool function below still supports it, but the agent's narration must say so explicitly
  rather than implying a meaningful country split exists.
- `current_assignee` is a **single pipe-delimited field mixing the assignee company AND inventor names**
  (e.g. `"GLAXO GROUP LIMITED|DUFFIELD, STEPHEN|ENEVER, CAROLYN|..."`), and some rows are an individual's
  name with no company at all (e.g. `"IACOBELLI STEFANO"`). We derive `primary_assignee` = the first
  pipe-segment as a heuristic proxy for "sponsor" — good enough for a "top N sponsors" ranking, but not a
  guaranteed clean company field. This heuristic + its limitation must be surfaced when relevant.
- Clinical `in_scope` is an object-dtype column of Python `True`/`False`/`NaN` (not a real bool dtype) —
  filter with `== True`, never the string `"True"`.


In [31]:
# Derived columns + known vocab (used both by the tool functions and to constrain the agent's
# function-calling schema so it can only pick real values, never invent a filter).
ip_landscape_df["primary_assignee"] = (
    ip_landscape_df["current_assignee"].fillna("").str.split("|").str[0].str.strip().replace("", None)
)
ip_whitespace_df["primary_assignee"] = (
    ip_whitespace_df["current_assignee"].fillna("").str.split("|").str[0].str.strip().replace("", None)
)

KNOWN_VOCAB = {
    "ip_modality_code": sorted(ip_landscape_df["modality_code"].dropna().unique().tolist()),
    "ip_authority": sorted(ip_landscape_df["authority"].dropna().unique().tolist()),
    "clinical_modality_code": sorted(clinical_df["modality_code"].dropna().unique().tolist()),
    "clinical_outcome": sorted(clinical_df["outcome"].dropna().unique().tolist()),
    "clinical_phase": sorted(clinical_df["phase"].dropna().unique().tolist()),
}
KNOWN_VOCAB


{'ip_modality_code': ['ADC', 'BISPECIFIC', 'BiTE', 'MAB', 'PROTAC'],
 'ip_authority': ['WO'],
 'clinical_modality_code': ['ADC',
  'ANTISENSE',
  'BISPECIFIC',
  'BiTE',
  'CAR-T',
  'DAC',
  'MAB',
  'MULTISPECIFIC',
  'OUT_OF_SCOPE',
  'PROTAC',
  'RADIOLIGAND',
  'UNCLASSIFIED'],
 'clinical_outcome': ['approved',
  'negative',
  'ongoing',
  'pending',
  'positive',
  'terminated',
  'unclear'],
 'clinical_phase': ['0',
  '1',
  '1-2',
  '2',
  '2-3',
  '3',
  '4',
  'EARLY_PHASE1',
  'Human pharmacology (Phase I): No Therapeutic exploratory (Phase II): No Therapeutic confirmatory - (Phase III): No Therapeutic use - (Phase IV): Yes',
  'Human pharmacology (Phase I): No Therapeutic exploratory (Phase II): No Therapeutic confirmatory - (Phase III): Yes Therapeutic use - (Phase IV): No',
  'Human pharmacology (Phase I): No Therapeutic exploratory (Phase II): Yes Therapeutic confirmatory - (Phase III): No Therapeutic use - (Phase IV): No',
  'Human pharmacology (Phase I): Yes Therapeuti

**Another data-quality finding:** the `phase` column vocab (above) is far messier than expected — most
rows are clean (`PHASE1`, `PHASE2`, `PHASE1|PHASE2`, ...) but a long tail is free text (`"Phase I,II"`,
verbose "Human pharmacology (Phase I): Yes ..." sentences, bare `"1"`/`"2"`, `"Not Applicable"`, etc).
Build a canonicalized `phase_group` (PHASE1/2/3/4/EARLY_PHASE1/UNKNOWN — furthest phase mentioned) so
"filter/sort by phase" queries don't silently drop or miscount the messy tail.


In [32]:
import re

_ROMAN_PHASE = {"IV": 4, "III": 3, "II": 2, "I": 1}

def canonicalize_phase(raw) -> str:
    """Best-effort furthest-phase canonicalization -> PHASE1/2/3/4, EARLY_PHASE1, or UNKNOWN.
    Handles: clean 'PHASE2'/'PHASE1|PHASE2' tokens, bare digits, roman numerals (whole word),
    and the verbose 'Human pharmacology (Phase I): Yes ...' sentence style (picks the highest
    phase marked yes)."""
    if pd.isna(raw) or not str(raw).strip():
        return "UNKNOWN"
    s = str(raw).upper()
    if "EARLY_PHASE1" in s or "EARLY PHASE 1" in s or "EARLY PHASE1" in s:
        return "EARLY_PHASE1"
    nums = set(int(n) for n in re.findall(r"PHASE\s*-?\s*([1-4])", s))
    # verbose EMA-style sentence: "...(Phase III): Yes..." -> only count phases marked YES
    for roman, yn in re.findall(r"\(?PHASE\s*([IV]{1,3})\)?\s*:?\s*(YES|NO)", s):
        if yn == "YES" and roman in _ROMAN_PHASE:
            nums.add(_ROMAN_PHASE[roman])
    if not nums:
        if s.strip() in {"1", "2", "3", "4"}:
            nums.add(int(s.strip()))
        else:
            for roman, val in _ROMAN_PHASE.items():
                if re.search(rf"\b{roman}\b", s):
                    nums.add(val)
                    break  # longest-first dict order avoids "I" matching inside "III"/"IV"
    return f"PHASE{max(nums)}" if nums else "UNKNOWN"

clinical_df["phase_group"] = clinical_df["phase"].apply(canonicalize_phase)
print(clinical_df["phase_group"].value_counts(dropna=False).to_dict())
KNOWN_VOCAB["clinical_phase_group"] = sorted(clinical_df["phase_group"].unique().tolist())


{'PHASE2': 10503, 'PHASE1': 4398, 'PHASE3': 2006, 'UNKNOWN': 1850, 'EARLY_PHASE1': 401, 'PHASE4': 199}


### Core tool functions

These are the ONLY things the LLM agent is ever allowed to call. Each returns a small, already-aggregated
pandas DataFrame (+ a plain-language note) — never a raw row dump of the full curated dataset, and every
filter is validated against `KNOWN_VOCAB` so the agent can't silently invent a nonexistent value.


In [33]:
class ToolError(Exception):
    """User-facing tool error (unknown column/value, no matching data, etc). The message is safe to
    show directly to the user -- tool functions never let a raw stack trace / file path escape."""
    pass


def _validate_choice(value, vocab_key, label):
    if value is None:
        return None
    vocab = KNOWN_VOCAB[vocab_key]
    match = next((v for v in vocab if v.upper() == str(value).upper()), None)
    if match is None:
        raise ToolError(f"Unknown {label} '{value}'. Known values: {', '.join(vocab)}")
    return match


# --- Tier 1: basic IP landscape queries (ip_final_version3.csv / app_data/ip_landscape_app.csv) -----
def top_n_sponsors(modality=None, indication_contains=None, n=10):
    """Top-N sponsors (assignee) by patent count."""
    modality = _validate_choice(modality, "ip_modality_code", "modality_code")
    df = ip_landscape_df
    if modality:
        df = df[df["modality_code"] == modality]
    if indication_contains:
        df = df[df["indications"].str.contains(re.escape(indication_contains), case=False, na=False)]
    if df.empty:
        raise ToolError("No IP records matched those filters.")
    counts = (df["primary_assignee"].dropna().value_counts().head(max(1, int(n)))
              .rename_axis("sponsor").reset_index(name="patent_count"))
    note = (f"Top {len(counts)} sponsors by patent count"
            + (f", modality={modality}" if modality else "")
            + (f", indication contains '{indication_contains}'" if indication_contains else "")
            + f" (n={len(df)} matching patents). 'sponsor' = primary assignee (first pipe-segment of "
              f"current_assignee) -- a heuristic; a few rows list an individual inventor, not a company.")
    return counts, note


_IP_SORT_COLS = ["filing_year", "application_date", "publication_date", "authority",
                  "primary_assignee", "target_harmonized", "indications"]


def sort_ip_patents(modality=None, sort_by="filing_year", ascending=True, indication_contains=None, limit=100):
    """Sort/filter IP patent rows by a column, optionally filtered by modality/indication."""
    modality = _validate_choice(modality, "ip_modality_code", "modality_code")
    if sort_by not in _IP_SORT_COLS:
        raise ToolError(f"Cannot sort by '{sort_by}'. Choose one of: {', '.join(_IP_SORT_COLS)}")
    df = ip_landscape_df
    if modality:
        df = df[df["modality_code"] == modality]
    if indication_contains:
        df = df[df["indications"].str.contains(re.escape(indication_contains), case=False, na=False)]
    if df.empty:
        raise ToolError("No IP records matched those filters.")
    cols = ["publication_number", "title", "primary_assignee", "filing_year", "authority",
            "modality_code", "target_harmonized", "indications"]
    out = df[cols].sort_values(sort_by, ascending=bool(ascending), na_position="last").head(int(limit))
    note = (f"{len(df)} {modality or 'all-modality'} patents sorted by {sort_by} "
            f"({'asc' if ascending else 'desc'}), showing top {len(out)}.")
    if sort_by == "authority" and df["authority"].nunique() <= 1:
        note += (" NOTE: this dataset's 'authority' field is almost entirely 'WO' (PCT international "
                 "filing) -- there is no meaningful per-country breakdown in the source data.")
    return out, note


def group_breakdown(dataset="ip", group_by="modality_code", filters=None, top_n=None):
    """Generic count-by-group breakdown over the ip or clinical basic-tier table."""
    df = {"ip": ip_landscape_df, "clinical": clinical_df}.get(dataset)
    if df is None:
        raise ToolError("dataset must be 'ip' or 'clinical'.")
    if group_by not in df.columns:
        raise ToolError(f"Unknown column '{group_by}' for dataset '{dataset}'.")
    for col, val in (filters or {}).items():
        if col not in df.columns:
            raise ToolError(f"Unknown filter column '{col}' for dataset '{dataset}'.")
        df = df[df[col] == val]
    if df.empty:
        raise ToolError("No records matched those filters.")
    counts = df[group_by].value_counts(dropna=False)
    if top_n:
        counts = counts.head(int(top_n))
    out = counts.rename_axis(group_by).reset_index(name="count")
    note = f"Count of {dataset} records grouped by {group_by} (n={len(df)} matching records)."
    return out, note


In [34]:
# --- Tier 1: cross-dataset comparison (IP vs clinical, for one target) ------------------------------
_PHASE_ORDER = {"PHASE4": 4, "PHASE3": 3, "PHASE2": 2, "PHASE1": 1, "EARLY_PHASE1": 0.5, "UNKNOWN": -1}


def compare_target_ip_vs_clinical(target_harmonized):
    """IP crowding snapshot vs clinical trial activity/outcomes for ONE harmonized target."""
    key = str(target_harmonized).strip().upper()
    ip_rows = ip_landscape_df[ip_landscape_df["target_harmonized"].str.upper() == key]
    clin_rows = clinical_df[clinical_df["target_harmonized"].str.upper() == key]
    if ip_rows.empty and clin_rows.empty:
        raise ToolError(f"No IP or clinical records found for target_harmonized='{target_harmonized}'.")
    outcome_counts = clin_rows["outcome"].value_counts().to_dict()
    furthest_phase = (max(clin_rows["phase_group"], key=lambda p: _PHASE_ORDER.get(p, -1))
                      if not clin_rows.empty else None)
    row = {
        "target_harmonized": target_harmonized,
        "patent_count": len(ip_rows),
        "ip_modalities": "|".join(sorted(ip_rows["modality_code"].dropna().unique())) or None,
        "top_ip_sponsors": ", ".join(ip_rows["primary_assignee"].value_counts().head(3).index.tolist()) or None,
        "earliest_filing_year": (int(ip_rows["filing_year"].min())
                                  if ip_rows["filing_year"].notna().any() else None),
        "trial_count": len(clin_rows),
        "clinical_modalities": "|".join(sorted(clin_rows["modality_code"].dropna().unique())) or None,
        "furthest_phase_reached": furthest_phase,
        "approved_trials": int(outcome_counts.get("approved", 0)),
        "positive_trials": int(outcome_counts.get("positive", 0)),
        "negative_trials": int(outcome_counts.get("negative", 0)),
        "terminated_trials": int(outcome_counts.get("terminated", 0)),
        "ongoing_or_pending_trials": int(outcome_counts.get("ongoing", 0) + outcome_counts.get("pending", 0)),
    }
    note = (f"IP vs clinical snapshot for target_harmonized='{target_harmonized}' "
            f"({len(ip_rows)} patents, {len(clin_rows)} trials).")
    return pd.DataFrame([row]), note


# --- Tier 2: 2x2 matrix / whitespace / triangulation (ip_final_version4.csv + MTW.run()) -------------
def run_whitespace_2x2(indication_or_tumor_type=None, target_harmonized=None, by_modality=False, sample_n=0):
    """2x2/triangulation placement via multitarget_locked_whitespace_workflow.MTW.run(), scoped to an
    indication/tumor type if given (pre-filters the clinical cohort, same pattern as the existing
    2x2 pilot/sweep notebooks) and/or to a single target."""
    terms = ([t.strip().lower() for t in re.split(r"[,/]", indication_or_tumor_type) if t.strip()]
             if indication_or_tumor_type else
             [t.strip().lower() for t in MTW.DEF_INDICATION.split(",")])
    clin_csv_path, tmp_clin_name = APP_CLINICAL_CSV, None
    if indication_or_tumor_type:
        mask = pd.Series(False, index=clinical_df.index)
        for col in ("m_conditions", "ctgov_conditions"):
            if col in clinical_df.columns:
                mask = mask | clinical_df[col].fillna("").str.lower().apply(
                    lambda s: any(t in s for t in terms))
        scoped = clinical_df[mask]
        if scoped.empty:
            raise ToolError(f"No clinical trials found matching indication/tumor-type terms {terms}.")
        tmp = tempfile.NamedTemporaryFile(mode="w", suffix=".csv", delete=False)
        scoped.to_csv(tmp.name, index=False)
        clin_csv_path, tmp_clin_name = tmp.name, tmp.name
    try:
        with tempfile.TemporaryDirectory() as tmp_out:
            rows = MTW.run(
                ip_csv=str(APP_IP_WHITESPACE_CSV), clin_csv=str(clin_csv_path),
                as_of=MTW.DEF_AS_OF, indication_terms=terms, outdir=tmp_out,
                key_col=MTW.DEF_KEY_COL, mod_col=MTW.DEF_MOD_COL,
                by_modality=bool(by_modality), sample_n=int(sample_n or 0), seed=7,
            )
    finally:
        if tmp_clin_name:
            os.unlink(tmp_clin_name)
    df = pd.DataFrame(rows)
    if target_harmonized:
        key = str(target_harmonized).strip().upper()
        df = df[df["target_harmonized"].str.upper() == key]
        if df.empty:
            raise ToolError(f"No 2x2/whitespace placement found for target_harmonized='{target_harmonized}'.")
    note = (f"2x2 whitespace/triangulation placement ({'target x modality' if by_modality else 'target-level'}), "
            f"indication/tumor-type terms={terms}, as-of {MTW.DEF_AS_OF}. Quadrants: TRUE WHITE SPACE "
            f"(validated biology, open IP), BATTLEGROUND (validated, contested IP), R&D TRAP (unproven, "
            f"open IP), RED FLAGS (unproven, contested). {len(df)} node(s) returned.")
    return df.sort_values(["quadrant", "clinical_performance_Y"], ascending=[True, False]), note


### Manual sanity checks of the tool functions (no LLM needed yet)

Reproduce the two example queries from the brief, plus one comparison and one 2x2/triangulation query.


In [35]:
# "give me the top 10 sponsors for bispecific antibody"
df1, note1 = top_n_sponsors(modality="BISPECIFIC", n=10)
print(note1)
display(df1)


Top 10 sponsors by patent count, modality=BISPECIFIC (n=2005 matching patents). 'sponsor' = primary assignee (first pipe-segment of current_assignee) -- a heuristic; a few rows list an individual inventor, not a company.


,sponsor,patent_count
0,F. HOFFMANN-LA ROCHE AG,120
1,"REGENERON PHARMACEUTICALS, INC.",74
2,"JANSSEN BIOTECH, INC.",60
3,"GENENTECH, INC.",56
4,AMGEN INC.,34
5,GENMAB A/S,32
6,"WUXI BIOLOGICS (SHANGHAI) CO., LTD.",22
7,MERUS N.V.,21
8,"XENCOR, INC.",20
9,CHUGAI SEIYAKU KABUSHIKI KAISHA,19


In [36]:
# "sort the ADC patents by source country"
df2, note2 = sort_ip_patents(modality="ADC", sort_by="authority", ascending=True, limit=15)
print(note2)
display(df2)


586 ADC patents sorted by authority (asc), showing top 15. NOTE: this dataset's 'authority' field is almost entirely 'WO' (PCT international filing) -- there is no meaningful per-country breakdown in the source data.


,publication_number,title,primary_assignee,filing_year,authority,modality_code,target_harmonized,indications
16,WO2010124797A1,Anti-mesothelin immunoconjugates and uses ther...,BAYER PHARMA AKTIENGESELLSCHAFT,2010,WO,ADC,Mesothelin-DM4,Lung | Mesothelioma
2173,WO2025027554A1,Humanized MUC1 antibody and antibody drug conj...,SUN PHARMA ADVANCED RESEARCH COMPANY LIMITED,2024,WO,ADC,MUC1-MMAE,Breast|Lung|Pancreatic|Gastric / GEJ|Liver / H...
2177,WO2025031307A1,Multidrug linker and antibody-drug conjugate,"SHANGHAI HUAO CO., LTD.",2024,WO,ADC,TROP-2,Kidney / RCC
2180,WO2025032079A1,Combination of a MPS1 inhibitor and an antibod...,NERVIANO MEDICAL SCIENCES S.R.L.,2024,WO,ADC,HER2,Colorectal|Breast|Lung|Prostate|Pancreatic|Gas...
2182,WO2025036307A1,Application of Anti-frα antibody-drug conjugat...,"BIO-THERA SOLUTIONS, LTD.",2024,WO,ADC,FOLR1,Lung
2183,WO2025036480A1,Compound and antibody-drug conjugate comprisin...,"AKESO BIOPHARMA CO., LTD.",2024,WO,ADC,HER3,Lung
2187,WO2025038638A1,Methods of treating cancer using Anti-her2 ant...,SEAGEN INC.,2024,WO,ADC,HER2-MMAE,Colorectal|Breast|Lung|Prostate|Pancreatic|Gas...
2172,WO2025026341A1,Anti-ROR1 protein antibody and conjugate thereof,"CSPC MEGALITH BIOPHARMACEUTICAL CO., LTD.",2024,WO,ADC,ROR1,Bladder / urothelial | Breast
2193,WO2025045015A1,Antibody-drug conjugate and preparation method...,"SICHUAN KELUN-BIOTECH BIOPHARMACEUTICAL CO., LTD.",2024,WO,ADC,HER3,Breast | Colorectal | Gastric / GEJ | Lung
2205,WO2025049926A1,Methods of selecting treatment options for cancer,YALE UNIVERSITY,2024,WO,ADC,HER2,Breast


In [37]:
# pick a real, well-populated target for the comparison + 2x2 sanity checks
top_targets = ip_landscape_df["target_harmonized"].value_counts().head(10)
print(top_targets)


target_harmonized
PD-1      278
PD-L1     178
HER2      156
EGFR      128
CD3        81
CTLA-4     73
B7-H3      66
CD47       61
TIGIT      49
TROP-2     48
Name: count, dtype: int64


In [38]:
# Comparison query: "compare IP crowding vs clinical progress for HER2"
df3, note3 = compare_target_ip_vs_clinical("HER2")
print(note3)
display(df3)


IP vs clinical snapshot for target_harmonized='HER2' (156 patents, 1620 trials).


,target_harmonized,patent_count,ip_modalities,top_ip_sponsors,earliest_filing_year,trial_count,clinical_modalities,furthest_phase_reached,approved_trials,positive_trials,negative_trials,terminated_trials,ongoing_or_pending_trials
0,HER2,156,ADC|BISPECIFIC|MAB,"SICHUAN KELUN-BIOTECH BIOPHARMACEUTICAL CO., L...",2010,1620,ADC|BISPECIFIC|CAR-T|MAB|OUT_OF_SCOPE|PROTAC|U...,PHASE4,4,66,26,181,1082


In [39]:
# 2x2/triangulation query: "give me the 2x2 whitespace matrix for pancreatic cancer targets"
df4, note4 = run_whitespace_2x2(indication_or_tumor_type="pancreatic", by_modality=False)
print(note4)
display(df4.head(20))


2x2 whitespace/triangulation placement (target-level), indication/tumor-type terms=['pancreatic'], as-of 2026-05-05. Quadrants: TRUE WHITE SPACE (validated biology, open IP), BATTLEGROUND (validated, contested IP), R&D TRAP (unproven, open IP), RED FLAGS (unproven, contested). 1242 node(s) returned.



MULTI-TARGET white-space (deterministic) — as of 2026-05-05 — mode: target (modality-collapsed)
    [ip] 4005 patent rows -> 1139 harmonized targets
    [clin] 660 trial rows -> 183 harmonized targets
    [out] /var/folders/k_/q9w49kk53zg7v019tx90gbjh0000gn/T/tmpax8fwll_/master_multitarget_whitespace.csv (1242 nodes)
    [out] /var/folders/k_/q9w49kk53zg7v019tx90gbjh0000gn/T/tmpax8fwll_/MAP_2x2_REPORT.md
    [out] /var/folders/k_/q9w49kk53zg7v019tx90gbjh0000gn/T/tmpax8fwll_/multitarget_2x2.png
done.


,target_harmonized,modality,quadrant,clinical_performance_Y,clinical_label,clinical_decisive_n,clinical_confidence,n_trials,epitope_crowding_X,x_status,fto_l1_epitope,fto_l2_format,fto_l3_use,fto_l1_lock_count,patents_grounded,ip_modalities,key_patent,blocking_assignee,x_resolved,y_resolved
393,CLDN18.2,(all),BATTLEGROUND,0.750,VALIDATED,4,1.00,19,0.651,weighted_fto_deterministic,0.651,0.795,0.232,20,27,ADC|BISPECIFIC|MAB,WO2025060679A1,"RENJI HOSPITAL, SHANGHAI JIAO TONG UNIVERSITY ...",True,True
82,B7-H3|CD3,(all),R&D TRAP,0.475,CONTESTED,1,0.33,1,0.000,weighted_fto_deterministic,0.000,0.000,0.000,1,1,BISPECIFIC,WO2021099347A1,DEUTSCHES KREBSFORSCHUNGSZENTRUM STIFTUNG DES ...,True,True
378,CEA|CEACAM5,(all),R&D TRAP,0.475,CONTESTED,1,0.33,2,0.000,weighted_fto_deterministic,0.000,0.000,0.000,1,1,BISPECIFIC,WO2024165403A1,PHILOGEN S.P.A.,True,True
400,CLDN18.2-MMAE,(all),R&D TRAP,0.475,CONTESTED,1,0.33,2,0.171,weighted_fto_deterministic,0.151,0.167,0.232,2,2,ADC,WO2025088105A1,"KEYMED BIOSCIENCES (CHENGDU) CO., LTD",True,True
741,IL-8,(all),R&D TRAP,0.475,CONTESTED,1,0.33,1,0.042,weighted_fto_deterministic,0.000,0.167,0.000,1,2,MAB,WO2024113122A1,"SUZHOU KAIGENE BIOTECHNOLOGY CO, LTD",True,True
950,PD-1|CD40,(all),R&D TRAP,0.475,CONTESTED,1,0.33,4,0.000,weighted_fto_deterministic,0.000,0.000,0.000,1,1,BISPECIFIC,WO2022078357A1,"BIOCYTOGEN PHARMACEUTICALS (BEIJING) CO., LTD",True,True
978,PD-1|VEGF,(all),R&D TRAP,0.475,CONTESTED,1,0.33,4,0.042,weighted_fto_deterministic,0.000,0.167,0.000,1,2,BISPECIFIC,WO2026064650A1,"PARAGON THERAPEUTICS, INC.",True,True
1071,ROR2,(all),R&D TRAP,0.475,CONTESTED,1,0.33,1,0.151,weighted_fto_deterministic,0.151,0.265,0.000,2,3,MAB,WO2024182475A2,"PHANES THERAPEUTICS, INC.",True,True
717,IGF-1R,(all),R&D TRAP,0.450,CONTESTED,1,0.33,2,0.349,weighted_fto_deterministic,0.349,0.469,0.000,5,7,BISPECIFIC|MAB,WO2016174053A1,PIERRE FABRE MEDICAMENT,True,True
857,MUC1,(all),R&D TRAP,0.450,CONTESTED,2,0.67,6,0.477,weighted_fto_deterministic,0.477,0.669,0.000,9,16,ADC|MAB,WO2025114381A1,DEUTSCHES KREBSFORSCHUNGSZENTRUM STIFTUNG DES ...,True,True


In [40]:
# DIAGNOSTIC: every node above came back x_resolved=False ("no grounded patents") -- check whether
# ip_final_version4.csv's claims-grounding columns are populated the way the engine expects.
for col in ["patsnap_claims_fulltext_status", "target_modality_crowding_proxy_basis",
            "claim_mentions_target", "claim_mentions_epitope_domain_or_competition"]:
    print(col, "->", ip_whitespace_df[col].value_counts(dropna=False).to_dict())
print()
print("Same columns in ip_landscape_df (v3), for comparison:")
for col in ["patsnap_claims_fulltext_status", "target_modality_crowding_proxy_basis",
            "claim_mentions_target", "claim_mentions_epitope_domain_or_competition"]:
    print(col, "->", ip_landscape_df[col].value_counts(dropna=False).to_dict())


patsnap_claims_fulltext_status -> {'pulled': 4004, 'api_error': 1}
target_modality_crowding_proxy_basis -> {nan: 2673, 'claim_target_and_modality': 1218, 'claim_target_only': 114}
claim_mentions_target -> {'yes': 4005}
claim_mentions_epitope_domain_or_competition -> {'yes': 2413, 'no': 1592}

Same columns in ip_landscape_df (v3), for comparison:
patsnap_claims_fulltext_status -> {'pulled': 8557, nan: 343, 'api_error': 1}
target_modality_crowding_proxy_basis -> {nan: 6107, 'claim_modality_only': 1321, 'claim_target_and_modality': 854, 'description_only': 440, 'claim_target_only': 105, 'not_detected': 74}
claim_mentions_target -> {'no': 4926, 'yes': 3632, nan: 343}
claim_mentions_epitope_domain_or_competition -> {'yes': 4365, 'no': 4193, nan: 343}


**BUG FOUND:** `application_date`/`filing_year` are all-digit strings in the raw CSVs (e.g. `20100330`,
`2010`), but the Phase 0 ETL read them with pandas' default type inference. Because these columns contain
some NaNs, pandas silently coerced them to `float64` and wrote them back out as `"20100330.0"` /
`"2010.0"`. `multitarget_locked_whitespace_workflow.py`'s date-cutoff helper (`W._ip_apd`) requires the
**whole string** to be digits (`d.isdigit()`) — the trailing `.0` breaks that check for every single row,
so every patent silently fails the "dated" gate and NO node ever resolves an X (IP/FTO) value. This is
exactly the kind of silent, hard-to-notice corruption the regression check earlier did NOT catch (that
check only sampled 40 nodes from the FULL, unscoped file — by coincidence none of them still exposed the
bug clearly enough to fail the assertion, since `x_resolved` differences weren't checked). **Fix:** re-read
the source CSVs with `dtype=str` (no type inference at all) so every value is byte-identical to the raw
file, and redo the trim + regression check properly this time (checking `x_resolved` explicitly, not just
the four spot-check fields).


In [41]:
# Double-check against the RAW (never touched by pandas) file first, before concluding this is purely
# an ETL-introduced bug -- run MTW against the ORIGINAL ip_final_version4.csv, full dataset, no scoping.
with tempfile.TemporaryDirectory() as tmp:
    rows_raw_full = MTW.run(
        ip_csv=str(RAW_IP_WHITESPACE_CSV), clin_csv=str(RAW_CLINICAL_CSV),
        as_of=MTW.DEF_AS_OF, indication_terms=[t.strip().lower() for t in MTW.DEF_INDICATION.split(",")],
        outdir=tmp, key_col=MTW.DEF_KEY_COL, mod_col=MTW.DEF_MOD_COL,
        by_modality=False, sample_n=0, seed=7,
    )
raw_full_df = pd.DataFrame(rows_raw_full)
print("x_resolved distribution on RAW untouched ip_final_version4.csv (full run):")
print(raw_full_df["x_resolved"].value_counts(dropna=False).to_dict())
print("x_status distribution:")
print(raw_full_df["x_status"].value_counts(dropna=False).to_dict())



MULTI-TARGET white-space (deterministic) — as of 2026-05-05 — mode: target (modality-collapsed)
    [ip] 4005 patent rows -> 1139 harmonized targets


x_resolved distribution on RAW untouched ip_final_version4.csv (full run):
{False: 2572}
x_status distribution:
{'UNRESOLVED_no_grounded_patents': 2572}


    [clin] 19357 trial rows -> 1691 harmonized targets
    [out] /var/folders/k_/q9w49kk53zg7v019tx90gbjh0000gn/T/tmp61apqs0x/master_multitarget_whitespace.csv (2572 nodes)
    [out] /var/folders/k_/q9w49kk53zg7v019tx90gbjh0000gn/T/tmp61apqs0x/MAP_2x2_REPORT.md
    [plot] skipped (no placed nodes)
done.


In [42]:
# So the failure is NOT an ETL artifact -- it reproduces from the untouched raw file too. Inspect the
# raw application_date/filing_year text directly via csv.DictReader (bypassing pandas entirely) to find
# the real root cause.
import csv as _csv
with open(RAW_IP_WHITESPACE_CSV, newline="", encoding="utf-8", errors="replace") as f:
    raw_rows = list(_csv.DictReader(f))
print("n rows:", len(raw_rows))
print("sample application_date/filing_year values:")
for r in raw_rows[:5]:
    print("  application_date=", repr(r.get("application_date")), " filing_year=", repr(r.get("filing_year")))
print()
apd_isdigit = sum(1 for r in raw_rows if (r.get("application_date") or "").strip().isdigit())
fy_isdigit = sum(1 for r in raw_rows if (r.get("filing_year") or "").strip().isdigit())
print(f"rows where application_date.isdigit(): {apd_isdigit} / {len(raw_rows)}")
print(f"rows where filing_year.isdigit(): {fy_isdigit} / {len(raw_rows)}")
print("distinct application_date lengths sample:", sorted(set(len((r.get('application_date') or '').strip()) for r in raw_rows[:200]))[:10])


n rows: 4005
sample application_date/filing_year values:
  application_date= '20100330.0'  filing_year= '2010.0'
  application_date= '20100330.0'  filing_year= '2010.0'
  application_date= '20100330.0'  filing_year= '2010.0'
  application_date= '20100416.0'  filing_year= '2010.0'
  application_date= '20100416.0'  filing_year= '2010.0'

rows where application_date.isdigit(): 0 / 4005
rows where filing_year.isdigit(): 0 / 4005
distinct application_date lengths sample: [10]


In [43]:
# ROOT CAUSE CONFIRMED: the trailing ".0" is ALREADY present in the raw, untouched ip_final_version4.csv
# on disk (not introduced by our ETL) -- application_date="20100330.0", filing_year="2010.0" straight
# from csv.DictReader. This silently breaks W._ip_apd's `.isdigit()` gate for every row in v4, so
# multitarget_locked_whitespace_workflow.py's X-axis (IP/FTO) can NEVER resolve against v4 as shipped.
# Check whether ip_final_version3.csv (the basic-landscape file, NOT the whitespace file, but sharing
# the same schema/lineage) has the same defect, using a lightweight streamed read (v3 is >50MB).
n_checked, n_bad_apd, n_bad_fy = 0, 0, 0
sample_vals = []
with open(RAW_IP_LANDSCAPE_CSV, newline="", encoding="utf-8", errors="replace") as f:
    for r in _csv.DictReader(f):
        n_checked += 1
        apd, fy = (r.get("application_date") or "").strip(), (r.get("filing_year") or "").strip()
        if apd and not apd.isdigit():
            n_bad_apd += 1
        if fy and not fy.isdigit():
            n_bad_fy += 1
        if len(sample_vals) < 5:
            sample_vals.append((apd, fy))
print("ip_final_version3.csv sample (application_date, filing_year):", sample_vals)
print(f"application_date non-digit: {n_bad_apd}/{n_checked}  |  filing_year non-digit: {n_bad_fy}/{n_checked}")


ip_final_version3.csv sample (application_date, filing_year): [('20100217', '2010'), ('20100223', '2010'), ('20100218', '2010'), ('20100305', '2010'), ('20100305', '2010')]
application_date non-digit: 0/8901  |  filing_year non-digit: 0/8901


**Real root cause found:** `ip_final_version3.csv` is clean (`20100217`, `2010`), but `ip_final_version4.csv`
already has the `.0`-suffixed corruption **baked into the file on disk**, independent of anything we did —
it must have been exported through a pandas step upstream that coerced these NaN-containing integer
columns to float before v4 was saved. As shipped, `ip_final_version4.csv` **cannot produce a single
resolved 2x2/whitespace placement** with `multitarget_locked_whitespace_workflow.py`, silently (no
exception — every node just quietly reports `UNRESOLVED_no_grounded_patents`). This is exactly the kind of
failure mode the plan's Phase 3 asked to hunt for. Fix: repair the trailing `.0` on the known numeric-ID
columns during Phase 0 trimming (read everything as plain text so we never introduce the same bug
ourselves, then strip a trailing `.0` off columns that are otherwise all-digits) — this is a safe,
targeted, auditable repair, not a guess at the underlying data.


## Phase 2 — Wiring the OpenAI "Data Analytics Agent"

The `.env` file lives at the project root (`../.env` relative to this notebook, i.e. NOT inside
`input/`, and NOT committed — see `.gitignore`) and holds `OPENAI_API_KEY`. We load it with
`python-dotenv` rather than requiring a shell `export`, so the key travels with the notebook kernel
regardless of how it was launched. **The key value itself is never printed, logged, or embedded in
any cell output below** — only boolean/length/prefix checks, matching the same discipline used
throughout this notebook for secrets.

Next: define the OpenAI function-calling schema for the 5 validated tool functions (constrained to
`KNOWN_VOCAB` enums), write the "10-15yr oncology/biologics data analytics expert" system prompt, and
wire a dispatch loop that only ever lets the LLM pick from these functions — it never sees or
generates freeform code/SQL against the raw data, and only ever receives the small aggregated
DataFrame results (not raw rows) to narrate.


In [44]:
from dotenv import load_dotenv
from openai import OpenAI

ENV_PATH = INPUT_DIR.parent / ".env"   # project root, NOT input/ -- gitignored, never committed
load_dotenv(dotenv_path=ENV_PATH)

_key = os.environ.get("OPENAI_API_KEY")
print(f".env path exists: {ENV_PATH.exists()}")
print(f"OPENAI_API_KEY loaded: {bool(_key)}  (length={len(_key) if _key else 0}, "
      f"prefix_ok={_key.startswith('sk-') if _key else False})")
assert _key, "OPENAI_API_KEY not found -- check the .env file at the project root."

client = OpenAI()  # picks up OPENAI_API_KEY from the environment automatically

# Minimal, cheap connectivity check (a couple of output tokens) -- proves the key is valid and the
# account can actually reach the Chat Completions API before we build anything on top of it.
_ping = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Reply with exactly one word: pong"}],
    max_tokens=5,
)
print("Connectivity check response:", _ping.choices[0].message.content)
del _key  # never keep the raw key value bound to a lingering notebook variable


.env path exists: True
OPENAI_API_KEY loaded: True  (length=164, prefix_ok=True)
Connectivity check response: pong


### Function-calling schema, persona, and dispatch loop

- Each of the 5 validated tool functions gets a strict JSON schema. Enum-constrained parameters
  (`modality`, `dataset`, `sort_by`, `outcome`, etc.) are pulled straight from `KNOWN_VOCAB` /
  `_IP_SORT_COLS`, so the LLM can only pick values that are guaranteed to be valid against the real
  data -- it cannot invent a modality or column name that doesn't exist.
- The LLM **never** sees raw rows. It only ever receives: (1) the tool schemas + persona system
  prompt, (2) the user's natural-language question, and (3) the small aggregated DataFrame (as
  `records` JSON) + explanatory note that a tool function returns. It writes the final narrative from
  that aggregated result only.
- Guardrail: every answer must echo back the interpreted filters/parameters before the narrative, so a
  wrong extraction is visible to the user rather than silently wrong (per the plan's accuracy
  principle).
- `ToolError` (raised by the tool functions for invalid filter values) is caught and fed back to the
  model as a tool result, not raised to the user -- the model can then explain the limitation or
  retry with a corrected value.


In [47]:
import json

TOOL_REGISTRY = {
    "top_n_sponsors": top_n_sponsors,
    "sort_ip_patents": sort_ip_patents,
    "group_breakdown": group_breakdown,
    "compare_target_ip_vs_clinical": compare_target_ip_vs_clinical,
    "run_whitespace_2x2": run_whitespace_2x2,
}

TOOLS_SCHEMA = [
    {"type": "function", "function": {
        "name": "top_n_sponsors",
        "description": "Top-N sponsors (assignee companies) ranked by patent count, optionally filtered "
                        "by modality and/or an indication keyword. Use for 'top N sponsors/companies for X'.",
        "parameters": {"type": "object", "properties": {
            "modality": {"type": ["string", "null"], "enum": KNOWN_VOCAB["ip_modality_code"],
                         "description": "Filter to one IP modality code, or omit for all modalities."},
            "indication_contains": {"type": ["string", "null"],
                                     "description": "Free-text substring to match against the indications field."},
            "n": {"type": "integer", "description": "How many top sponsors to return.", "default": 10},
        }, "required": []},
    }},
    {"type": "function", "function": {
        "name": "sort_ip_patents",
        "description": "Sort/filter IP patent rows by a column (e.g. filing year, authority/country, "
                        "assignee, target). Use for 'sort/list the X patents by Y'.",
        "parameters": {"type": "object", "properties": {
            "modality": {"type": ["string", "null"], "enum": KNOWN_VOCAB["ip_modality_code"]},
            "sort_by": {"type": "string", "enum": _IP_SORT_COLS, "default": "filing_year"},
            "ascending": {"type": "boolean", "default": True},
            "indication_contains": {"type": ["string", "null"],
                                     "description": "Free-text substring to match against the indications field."},
            "limit": {"type": "integer", "default": 100},
        }, "required": ["sort_by"]},
    }},
    {"type": "function", "function": {
        "name": "group_breakdown",
        "description": "Count-by-group breakdown of the IP or clinical dataset (e.g. patents by authority, "
                        "trials by outcome/phase_group). Use for 'how many X by Y' / distribution questions.",
        "parameters": {"type": "object", "properties": {
            "dataset": {"type": "string", "enum": ["ip", "clinical"]},
            "group_by": {"type": "string",
                         "description": "Column to group by. For dataset='ip', typical choices: "
                                        "modality_code, authority, target_harmonized, primary_assignee, "
                                        "filing_year. For dataset='clinical', typical choices: modality_code, "
                                        "outcome, phase_group, target_harmonized, lead_sponsor."},
            "filters": {"type": ["object", "null"],
                        "description": "Optional exact-match column:value filters, e.g. {\"modality_code\": \"ADC\"}."},
            "top_n": {"type": ["integer", "null"]},
        }, "required": ["dataset", "group_by"]},
    }},
    {"type": "function", "function": {
        "name": "compare_target_ip_vs_clinical",
        "description": "Cross-dataset snapshot for ONE harmonized target: IP crowding (patent count, top "
                        "sponsors, modalities) vs clinical activity/outcomes (trial count, furthest phase, "
                        "approved/positive/negative/terminated counts). Use for 'compare IP vs clinical "
                        "progress for target X'.",
        "parameters": {"type": "object", "properties": {
            "target_harmonized": {"type": "string", "description": "Harmonized target symbol, e.g. HER2, EGFR, CLDN18.2."},
        }, "required": ["target_harmonized"]},
    }},
    {"type": "function", "function": {
        "name": "run_whitespace_2x2",
        "description": "2x2 whitespace/triangulation matrix (epitope crowding X-axis vs clinical performance "
                        "Y-axis) placing targets into TRUE WHITE SPACE / BATTLEGROUND / R&D TRAP / RED FLAGS "
                        "quadrants. Use for '2x2', 'whitespace', 'triangulation' questions, optionally scoped "
                        "to an indication/tumor type and/or a single target.",
        "parameters": {"type": "object", "properties": {
            "indication_or_tumor_type": {"type": ["string", "null"],
                                         "description": "e.g. 'pancreatic cancer'. Omit for the workflow's default indication scope."},
            "target_harmonized": {"type": ["string", "null"],
                                  "description": "Restrict the result to one target's placement, or omit for all nodes."},
            "by_modality": {"type": "boolean", "default": False,
                            "description": "True = place target x modality combinations separately."},
            "sample_n": {"type": "integer", "default": 0, "description": "0 = no subsampling of the node grid."},
        }, "required": []},
    }},
]

SYSTEM_PROMPT = """You are the Data Analytics Agent for an oncology antibody-therapeutics IP and clinical \
intelligence tool, used by board members. You have 10-15 years of hands-on experience in oncology \
biologics R&D, patent/IP strategy, and clinical development -- you understand modalities (mAb, ADC, \
bispecific, BiTE, CAR-T, radioligand, PROTAC), trial phases, and patent landscaping conventions.

Rules you must always follow:
1. You never write or execute freeform code/SQL against the raw data. You may ONLY answer by calling one \
of the provided tool functions. If no tool fits the question, say so plainly instead of guessing.
2. You never see or fabricate raw data rows -- you only receive small, already-aggregated tool results. \
Narrate ONLY from those results; never invent numbers, sponsor names, or targets not present in the result.
3. Always begin your final answer with a short "Interpreted as:" line stating the filters/parameters you \
extracted from the user's question (modality, indication, target, sort column, etc.), so a wrong \
interpretation is visible rather than silently wrong.
4. If a tool call fails (unknown value, no matching data), explain the limitation plainly to the user \
(e.g. list the valid known values if given) rather than retrying blindly or apologizing excessively.
5. Preserve important data-quality caveats surfaced by the tools verbatim (e.g. the patent 'authority' \
field being almost entirely WO/PCT filings with no real per-country breakdown, or 'sponsor' being a \
heuristic first-segment of a multi-assignee field) -- do not smooth these over.
6. Be concise and board-appropriate: lead with the headline finding, then supporting detail."""


def _dispatch_tool_call(name, args_json):
    """Executes one LLM-requested tool call against the real Python functions, returning a JSON-safe
    string result. ToolError becomes a normal (non-crashing) error message fed back to the model."""
    fn = TOOL_REGISTRY.get(name)
    if fn is None:
        return json.dumps({"error": f"Unknown tool '{name}'."})
    try:
        args = json.loads(args_json or "{}")
        args = {k: v for k, v in args.items() if v is not None}
        result_df, note = fn(**args)
        return json.dumps({
            "note": note,
            "columns": list(result_df.columns),
            "rows": json.loads(result_df.head(50).to_json(orient="records")),
            "row_count_returned": len(result_df),
        })
    except ToolError as e:
        return json.dumps({"error": str(e)})
    except TypeError as e:
        return json.dumps({"error": f"Invalid arguments for '{name}': {e}"})


def ask_agent(user_query, model="gpt-4o-mini", max_rounds=4):
    """Runs the full NLQ -> tool-call -> narration loop for one user question. Returns the final
    narrated answer string. Prints each tool call + a short result summary for visibility during testing."""
    messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user_query}]
    for round_idx in range(max_rounds):
        # Force a tool call on the first round -- otherwise a small model can narrate its *intent*
        # ("I will now retrieve...") instead of actually invoking a function. Subsequent rounds go back
        # to "auto" so the model can freely choose to stop and narrate once it has a result.
        tool_choice = "required" if round_idx == 0 else "auto"
        resp = client.chat.completions.create(
            model=model, messages=messages, tools=TOOLS_SCHEMA, tool_choice=tool_choice)
        msg = resp.choices[0].message
        if not msg.tool_calls:
            return msg.content
        messages.append(msg)
        for tc in msg.tool_calls:
            print(f"  [tool call] {tc.function.name}({tc.function.arguments})")
            result = _dispatch_tool_call(tc.function.name, tc.function.arguments)
            print(f"  [tool result] {result[:300]}{'...' if len(result) > 300 else ''}")
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})
    return "Agent did not converge to a final answer within the round limit."


### End-to-end NLQ tests (Phase 1 verification requirement)

Testing, in order: (1) the two example prompts from the original brief, (2) a cross-dataset comparison
prompt, (3) a 2x2/whitespace/triangulation prompt. Each print shows the tool call(s) the model chose, a
truncated preview of the tool result fed back to it, and the final narrated answer (which must open with
"Interpreted as:").


In [48]:
print("=" * 90)
print("TEST 1: top_n_sponsors")
print("=" * 90)
answer1 = ask_agent("Give me the top 10 sponsors for bispecific antibody")
print("\nFINAL ANSWER:\n", answer1)


TEST 1: top_n_sponsors
  [tool call] top_n_sponsors({"modality":"BISPECIFIC","n":10})
  [tool result] {"note": "Top 10 sponsors by patent count, modality=BISPECIFIC (n=2005 matching patents). 'sponsor' = primary assignee (first pipe-segment of current_assignee) -- a heuristic; a few rows list an individual inventor, not a company.", "columns": ["sponsor", "patent_count"], "rows": [{"sponsor": "F. HO...

FINAL ANSWER:
 Interpreted as: Top 10 sponsors for bispecific antibodies.

1. **F. HOFFMANN-LA ROCHE AG** - 120 patents
2. **REGENERON PHARMACEUTICALS, INC.** - 74 patents
3. **JANSSEN BIOTECH, INC.** - 60 patents
4. **GENENTECH, INC.** - 56 patents
5. **AMGEN INC.** - 34 patents
6. **GENMAB A/S** - 32 patents
7. **WUXI BIOLOGICS (SHANGHAI) CO., LTD.** - 22 patents
8. **MERUS N.V.** - 21 patents
9. **XENCOR, INC.** - 20 patents
10. **CHUGAI SEIYAKU KABUSHIKI KAISHA** - 19 patents

**Note:** The 'sponsor' reflects the primary assignee determined from the first segment of a multi-assignee

In [49]:
print("=" * 90)
print("TEST 2: sort_ip_patents (should surface the WO-only authority caveat)")
print("=" * 90)
answer2 = ask_agent("Sort the ADC patents by source country")
print("\nFINAL ANSWER:\n", answer2)


TEST 2: sort_ip_patents (should surface the WO-only authority caveat)
  [tool call] sort_ip_patents({"modality":"ADC","sort_by":"authority"})
  [tool result] {"note": "586 ADC patents sorted by authority (asc), showing top 100. NOTE: this dataset's 'authority' field is almost entirely 'WO' (PCT international filing) -- there is no meaningful per-country breakdown in the source data.", "columns": ["publication_number", "title", "primary_assignee", "filing...

FINAL ANSWER:
 Interpreted as: sorting ADC patents by source country (authority).

The sorted list of ADC patents reveals that the 'authority' field primarily indicates 'WO' (PCT international filing) for the entire dataset. As a result, there is no meaningful per-country breakdown in the source data. Here are the top 100 ADC patents:

1. **Publication Number:** WO2010124797A1  
   **Title:** Anti-mesothelin immunoconjugates and uses therefor  
   **Primary Assignee:** BAYER PHARMA AKTIENGESELLSCHAFT  
   **Filing Year:** 2010  
  

In [50]:
print("=" * 90)
print("TEST 3: compare_target_ip_vs_clinical")
print("=" * 90)
answer3 = ask_agent("Compare IP crowding vs clinical progress for HER2")
print("\nFINAL ANSWER:\n", answer3)


TEST 3: compare_target_ip_vs_clinical
  [tool call] compare_target_ip_vs_clinical({"target_harmonized":"HER2"})
  [tool result] {"note": "IP vs clinical snapshot for target_harmonized='HER2' (156 patents, 1620 trials).", "columns": ["target_harmonized", "patent_count", "ip_modalities", "top_ip_sponsors", "earliest_filing_year", "trial_count", "clinical_modalities", "furthest_phase_reached", "approved_trials", "positive_trial...

FINAL ANSWER:
 Interpreted as: Comparing IP crowding vs clinical progress for the target HER2.

- **Patent Landscape**: There are 156 patents related to HER2, with modalities including ADC, bispecific, and mAb. Top sponsors are Sichuan Kelun-Biotech Biopharmaceutical Co., Ltd., Chia Tai Tianqing Pharmaceutical Group Co., Ltd., and Bolt Biotherapeutics, Inc. The earliest filing year is 2010.

- **Clinical Activity**: A total of 1,620 clinical trials have been associated with HER2, spanning modalities including ADC, bispecific, CAR-T, mAb, PROTAC, and others labe

In [ ]:
print("=" * 90)
print("TEST 4: run_whitespace_2x2")
print("=" * 90)
answer4 = ask_agent("Show me the 2x2 whitespace matrix for pancreatic cancer")
print("\nFINAL ANSWER:\n", answer4)
